In [ ]:
!pip install ultralytics -q

In [ ]:
import torch
print("GPU tersedia:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tidak ada GPU")

In [ ]:
# Car Detection & Tracking dengan YOLOv8
# menggunakan: Ultralytics YOLOv8 + OpenCV

import cv2
from ultralytics import YOLO
from IPython.display import Video, display
import os

INPUT_VIDEO  = "test-video.mp4"
OUTPUT_RAW   = "output_raw.avi"
OUTPUT_FINAL = "output_final.mp4"

# class ID 'car' pada dataset COCO yang dipakai YOLOv8
CAR_CLASS_ID = 2
CONFIDENCE_THRESHOLD = 0.35


# fungsi 1: Muat model YOLOv8
def load_model(model_name: str = "yolov8n.pt") -> YOLO:
    print(f"[info] mmuat model: {model_name}")
    model = YOLO(model_name)
    print(f"[info] model berhasil dimuat!")
    return model

# fungsi 2: filter hasil deteksi — hanya ambil mobil
def filter_cars(results, conf_threshold: float = CONFIDENCE_THRESHOLD):
    cars = []

    for box in results[0].boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        if class_id == CAR_CLASS_ID and confidence >= conf_threshold:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            track_id = int(box.id[0]) if box.id is not None else None
            cars.append({
                "box": (x1, y1, x2, y2),
                "conf": confidence,
                "track_id": track_id
            })
    return cars

# fungsi 3: gambar bounding box pada frame
def draw_detections(frame, detections: list):
    """
    menggambar bounding box dan label pada frame video.
    hanya objek dari list 'detections' (sudah difilter) yang digambar.
    """
    for det in detections:
        x1, y1, x2, y2 = det["box"]
        conf = det["conf"]
        track_id = det["track_id"]
        color = (0, 255, 0)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        if track_id is not None:
            label = f"Car #{track_id} | {conf:.0%}"  # Contoh: "Car #3 | 87%"
        else:
            label = f"Car | {conf:.0%}"

        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - text_h - 8), (x1 + text_w, y1), (0, 0, 0), -1)
        cv2.putText(frame, label, (x1, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    return frame

# fungsi 4: proses video — fungsi utama pipeline
def process_video(input_path: str, output_path: str, model: YOLO):
    """
    pipeline utama: baca video → deteksi+tracking → filter mobil → tulis output.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"[error] video tidak ditemukan: {input_path}")

    fps    = int(cap.get(cv2.CAP_PROP_FPS))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"[info] video input  : {input_path}")
    print(f"[info] resolusi     : {width}x{height} @ {fps} FPS")
    print(f"[info] total frame  : {total}")

    fourcc = cv2.VideoWriter_fourcc(*"XVID")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_count = 0

    print("[info] mulai memproses video...\n")

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        results = model.track(frame, persist=True, verbose=False)
        car_detections = filter_cars(results)
        annotated_frame = draw_detections(frame, car_detections)
        out.write(annotated_frame)
        frame_count += 1

        if frame_count % 30 == 0:
            pct = (frame_count / total * 100) if total > 0 else 0
            print(f"  →frame {frame_count}/{total} ({pct:.1f}%)")
    cap.release()
    out.release()

    print(f"\n[info] selesai! output disimpan: {output_path}")
    print(f"[info] total frame diproses: {frame_count}")

# FUNGSI 5: Konversi ke MP4 H.264 & tampilkan di Colab

def visualize_video(raw_path: str, final_path: str):
    """
    mengkonversi video AVI ke mp4 H.264 menggunakan ffmpeg,
    lalu menampilkannya langsung di dalam notebook Google Colab.

    kenapa perlu konversi?
    → openCV menghasilkan .avi yang tidak bisa langsung diputar di browser/Colab.
    → ffmpeg mengkonversi ke H.264/mp4 yang didukung semua browser modern.
    """
    print(f"[info] mengkonversi {raw_path} → {final_path} ...")
    os.system(f"ffmpeg -y -i {raw_path} -vcodec libx264 -acodec aac -strict -2 {final_path} -loglevel quiet")

    if os.path.exists(final_path):
        size_mb = os.path.getsize(final_path) / (1024 * 1024)
        print(f"[info] konversi berhasil! ukuran file: {size_mb:.2f} mb")
        print("[info] menampilkan video...\n")
        display(Video(final_path, embed=True, width=720))
    else:
        print("[error] konversi gagal. pastikan ffmpeg terinstall.")

# main — jalankan seluruh pipeline

if __name__ == "__main__" or True:
    # step 1: muat model YOLOv8 nano
    model = load_model("yolov8n.pt")

    # step 2: proses video — deteksi & tracking mobil, simpan output mentah
    process_video(INPUT_VIDEO, OUTPUT_RAW, model)

    # step 3: konversi ke H.264 mp4 dan tampilkan di notebook
    visualize_video(OUTPUT_RAW, OUTPUT_FINAL)